## Daily adjusted price histories

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

prices = pd.read_csv(Path("data") / "etf_daily_joint.csv", parse_dates=["date"], index_col="date")
fig, axes = plt.subplots(4, 2, figsize=(14, 13), sharex=True, constrained_layout=True)
for ticker, ax in zip(["QLD", "QQQ", "UBT", "TLT", "USD", "ROM", "IWM", "UWM"], axes.flat):
    ax.plot(prices.index, prices[f"{ticker}_adj_close"], linewidth=1)
    ax.set_title(ticker)
    ax.grid(alpha=0.2)
plt.show()


In [ ]:
import numpy as np
from matplotlib.ticker import PercentFormatter

def drawdown(growth):
    return growth / growth.cummax().clip(lower=1.0) - 1

def plot_equity_drawdown(growth, title):
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1]}, constrained_layout=True)
    growth.plot(ax=axes[0], logy=True, grid=True)
    axes[0].set_title(title)
    axes[0].set_ylabel("Growth of 1")
    dd = drawdown(growth)
    for column, line in zip(growth.columns, axes[0].lines):
        axes[1].plot(dd.index, dd[column], color=line.get_color(), linewidth=1)
    axes[1].yaxis.set_major_formatter(PercentFormatter(1))
    axes[1].set_ylabel("Drawdown")
    axes[1].grid(alpha=0.3)
    plt.show()


## Test 01: QQQ/TLT spread RSI, trading QQQ and TLT at the next open

In [ ]:
spread = prices["QQQ_adj_close"] / prices["TLT_adj_close"]
spread.plot(figsize=(14, 4), linewidth=1, grid=True, title="QQQ / TLT adjusted close spread")
plt.show()


In [ ]:
change = spread.diff()
gain = change.clip(lower=0)
loss = -change.clip(upper=0)
avg_gain = gain.iloc[1:4].mean()
avg_loss = loss.iloc[1:4].mean()
rsi = pd.Series(float("nan"), index=prices.index, name="rsi_3")
for i in range(3, len(prices)):
    if i > 3:
        avg_gain = (2 * avg_gain + gain.iloc[i]) / 3
        avg_loss = (2 * avg_loss + loss.iloc[i]) / 3
    if avg_gain == 0 and avg_loss == 0:
        rsi.iloc[i] = 50
    elif avg_loss == 0:
        rsi.iloc[i] = 100
    else:
        rsi.iloc[i] = 100 - 100 / (1 + avg_gain / avg_loss)

rsi.plot(figsize=(14, 4), linewidth=1, title="RSI (3), Wilder smoothing")
for level in [15, 30, 70, 85]:
    plt.axhline(level, linestyle="--", alpha=0.4)
plt.show()


In [ ]:
target = pd.Series("CASH", index=prices.index, name="close_signal_target")
holding = "CASH"
for date, value in rsi.items():
    if pd.notna(value):
        if holding == "QQQ" and value > 70:
            holding = "CASH"
        elif holding == "TLT" and value < 30:
            holding = "CASH"
        if holding == "CASH":
            if value < 15:
                holding = "QQQ"
            elif value > 85:
                holding = "TLT"
    target.loc[date] = holding

position = target.shift(1, fill_value="CASH").rename("position_after_open")
previous_position = position.shift(1, fill_value="CASH")
signal_date = pd.Series(prices.index, index=prices.index).shift(1)
changed = position.ne(previous_position)
trades = pd.DataFrame({
    "signal_date": signal_date,
    "execution_date": prices.index,
    "from": previous_position,
    "to": position,
}).loc[changed]
assert (trades["signal_date"] < trades["execution_date"]).all()
trades


In [ ]:
close = prices[["QQQ_adj_close", "TLT_adj_close"]].copy()
close.columns = ["QQQ", "TLT"]
open_price = pd.DataFrame({
    ticker: prices[f"{ticker}_open"] * prices[f"{ticker}_adj_close"] / prices[f"{ticker}_close"]
    for ticker in ["QQQ", "TLT"]
})
overnight = open_price / close.shift(1) - 1
intraday = close / open_price - 1
strategy_overnight = pd.Series(0.0, index=prices.index)
strategy_intraday = pd.Series(0.0, index=prices.index)
for ticker in ["QQQ", "TLT"]:
    held_overnight = previous_position.eq(ticker)
    held_intraday = position.eq(ticker)
    strategy_overnight.loc[held_overnight] = overnight.loc[held_overnight, ticker]
    strategy_intraday.loc[held_intraday] = intraday.loc[held_intraday, ticker]
strategy_return = (1 + strategy_overnight) * (1 + strategy_intraday) - 1
assert strategy_return.notna().all()
equity = (1 + strategy_return).cumprod()
results = pd.DataFrame({
    "spread": spread,
    "rsi_3": rsi,
    "close_signal_target": target,
    "position_after_open": position,
    "strategy_return_before_costs": strategy_return,
    "equity": equity,
})
results.tail()


In [ ]:
comparison = close / close.iloc[0]
comparison["Strategy"] = equity
plot_equity_drawdown(comparison, "QQQ / TLT: next-open execution, before costs, cash earns 0%")


## Test 02: QQQ/TLT spread RSI, trading QLD/UBT; entries 10/90, exits 60/40; next-open execution

In [ ]:
leveraged_target = pd.Series("CASH", index=prices.index)
holding = "CASH"
for date, value in rsi.items():
    if pd.notna(value):
        if holding == "QLD" and value > 60:
            holding = "CASH"
        elif holding == "UBT" and value < 40:
            holding = "CASH"
        if holding == "CASH":
            if value < 10:
                holding = "QLD"
            elif value > 90:
                holding = "UBT"
    leveraged_target.loc[date] = holding
leveraged_position = leveraged_target.shift(1, fill_value="CASH")
leveraged_previous_position = leveraged_position.shift(1, fill_value="CASH")
leveraged_trades = pd.DataFrame({
    "signal_date": pd.Series(prices.index, index=prices.index).shift(1),
    "execution_date": prices.index,
    "from": leveraged_previous_position,
    "to": leveraged_position,
}).loc[leveraged_position.ne(leveraged_previous_position)]
assert (leveraged_trades.signal_date < leveraged_trades.execution_date).all()
leveraged_trades


In [ ]:
leveraged_close = prices[["QLD_adj_close", "UBT_adj_close"]].copy()
leveraged_close.columns = ["QLD", "UBT"]
leveraged_open = pd.DataFrame({
    ticker: prices[f"{ticker}_open"] * prices[f"{ticker}_adj_close"] / prices[f"{ticker}_close"]
    for ticker in ["QLD", "UBT"]
})
leveraged_overnight = leveraged_open / leveraged_close.shift(1) - 1
leveraged_intraday = leveraged_close / leveraged_open - 1
leveraged_strategy_overnight = pd.Series(0.0, index=prices.index)
leveraged_strategy_intraday = pd.Series(0.0, index=prices.index)
for ticker in ["QLD", "UBT"]:
    held_overnight = leveraged_previous_position.eq(ticker)
    held_intraday = leveraged_position.eq(ticker)
    leveraged_strategy_overnight.loc[held_overnight] = leveraged_overnight.loc[held_overnight, ticker]
    leveraged_strategy_intraday.loc[held_intraday] = leveraged_intraday.loc[held_intraday, ticker]
leveraged_strategy_return = (1 + leveraged_strategy_overnight) * (1 + leveraged_strategy_intraday) - 1
assert leveraged_strategy_return.notna().all()
leveraged_equity = (1 + leveraged_strategy_return).cumprod()
leveraged_results = pd.DataFrame({
    "spread": spread,
    "rsi_3": rsi,
    "close_signal_target": leveraged_target,
    "position_after_open": leveraged_position,
    "strategy_return_before_costs": leveraged_strategy_return,
    "equity": leveraged_equity,
})
leveraged_results.tail()


In [ ]:
leveraged_comparison = leveraged_close / leveraged_close.iloc[0]
leveraged_comparison["QLD / UBT strategy"] = leveraged_equity
leveraged_comparison["QQQ / TLT strategy"] = equity
plot_equity_drawdown(leveraged_comparison, "QLD / UBT: QQQ/TLT signal, before costs, cash earns 0%")


## Test 03: USD/UBT and ROM/UBT spread RSI; entries 10/90, exits 60/40; next-open execution

In [ ]:
def wilder_rsi(series, period=3):
    change = series.diff()
    gain = change.clip(lower=0)
    loss = -change.clip(upper=0)
    avg_gain = gain.iloc[1:period + 1].mean()
    avg_loss = loss.iloc[1:period + 1].mean()
    values = pd.Series(float("nan"), index=series.index)
    for i in range(period, len(series)):
        if i > period:
            avg_gain = ((period - 1) * avg_gain + gain.iloc[i]) / period
            avg_loss = ((period - 1) * avg_loss + loss.iloc[i]) / period
        if avg_gain == 0 and avg_loss == 0:
            values.iloc[i] = 50
        elif avg_loss == 0:
            values.iloc[i] = 100
        else:
            values.iloc[i] = 100 - 100 / (1 + avg_gain / avg_loss)
    return values

sector_spreads = pd.DataFrame({
    ticker: prices[f"{ticker}_adj_close"] / prices["UBT_adj_close"]
    for ticker in ["USD", "ROM"]
})
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True, constrained_layout=True)
for ticker, ax in zip(["USD", "ROM"], axes):
    ax.plot(sector_spreads[ticker], linewidth=1)
    ax.set_title(f"{ticker} / UBT adjusted close spread")
    ax.grid(alpha=0.2)
plt.show()

sector_rsi = sector_spreads.apply(wilder_rsi)
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True, constrained_layout=True)
for ticker, ax in zip(["USD", "ROM"], axes):
    ax.plot(sector_rsi[ticker], linewidth=1)
    ax.set_title(f"{ticker} / UBT RSI (3)")
    for level in [10, 40, 60, 90]:
        ax.axhline(level, linestyle="--", alpha=0.4)
plt.show()


In [ ]:
sector_results = {}
sector_trades = {}
for ticker in ["USD", "ROM"]:
    pair_target = pd.Series("CASH", index=prices.index)
    holding = "CASH"
    for date, value in sector_rsi[ticker].items():
        if pd.notna(value):
            if holding == ticker and value > 60:
                holding = "CASH"
            elif holding == "UBT" and value < 40:
                holding = "CASH"
            if holding == "CASH":
                if value < 10:
                    holding = ticker
                elif value > 90:
                    holding = "UBT"
        pair_target.loc[date] = holding
    pair_position = pair_target.shift(1, fill_value="CASH")
    pair_previous = pair_position.shift(1, fill_value="CASH")
    pair_overnight = pd.Series(0.0, index=prices.index)
    pair_intraday = pd.Series(0.0, index=prices.index)
    for asset in [ticker, "UBT"]:
        asset_close = prices[f"{asset}_adj_close"]
        asset_open = prices[f"{asset}_open"] * asset_close / prices[f"{asset}_close"]
        held_overnight = pair_previous.eq(asset)
        held_intraday = pair_position.eq(asset)
        pair_overnight.loc[held_overnight] = (asset_open / asset_close.shift(1) - 1).loc[held_overnight]
        pair_intraday.loc[held_intraday] = (asset_close / asset_open - 1).loc[held_intraday]
    pair_return = (1 + pair_overnight) * (1 + pair_intraday) - 1
    assert pair_return.notna().all()
    sector_results[ticker] = pd.DataFrame({
        "spread": sector_spreads[ticker],
        "rsi_3": sector_rsi[ticker],
        "close_signal_target": pair_target,
        "position_after_open": pair_position,
        "strategy_return_before_costs": pair_return,
        "equity": (1 + pair_return).cumprod(),
    })
    sector_trades[ticker] = pd.DataFrame({
        "signal_date": pd.Series(prices.index, index=prices.index).shift(1),
        "execution_date": prices.index,
        "from": pair_previous,
        "to": pair_position,
    }).loc[pair_position.ne(pair_previous)]
    assert (sector_trades[ticker].signal_date < sector_trades[ticker].execution_date).all()


In [ ]:
for ticker in ["USD", "ROM"]:
    pair_comparison = pd.DataFrame({
        f"{ticker} / UBT strategy": sector_results[ticker]["equity"],
        f"{ticker} buy and hold": prices[f"{ticker}_adj_close"] / prices[f"{ticker}_adj_close"].iloc[0],
        "UBT buy and hold": prices["UBT_adj_close"] / prices["UBT_adj_close"].iloc[0],
    })
    plot_equity_drawdown(pair_comparison, f"{ticker} / UBT: next-open execution, before costs, cash earns 0%")


## Test 04: IWM/TLT spread RSI; trade IWM/TLT (15/85, 70/30) or UWM/UBT (10/90, 60/40); next-open execution

In [ ]:
russell_spread = prices["IWM_adj_close"] / prices["TLT_adj_close"]
russell_spread.plot(figsize=(14, 4), linewidth=1, grid=True, title="IWM / TLT adjusted close spread")
plt.show()


In [ ]:
russell_rsi = wilder_rsi(russell_spread, 3)
russell_rsi.plot(figsize=(14, 4), linewidth=1, title="IWM / TLT RSI (3), Wilder smoothing")
for level in [10, 15, 30, 40, 60, 70, 85, 90]:
    plt.axhline(level, linestyle="--", alpha=0.3)
plt.show()


In [ ]:
russell_tests = {
    "IWM / TLT": ("IWM", "TLT", 15, 85, 70, 30),
    "UWM / UBT": ("UWM", "UBT", 10, 90, 60, 40),
}
russell_results = {}
russell_trades = {}
for name, (risk, bond, entry_low, entry_high, exit_risk, exit_bond) in russell_tests.items():
    pair_target = pd.Series("CASH", index=prices.index)
    holding = "CASH"
    for date, value in russell_rsi.items():
        if pd.notna(value):
            if holding == risk and value > exit_risk:
                holding = "CASH"
            elif holding == bond and value < exit_bond:
                holding = "CASH"
            if holding == "CASH":
                if value < entry_low:
                    holding = risk
                elif value > entry_high:
                    holding = bond
        pair_target.loc[date] = holding
    pair_position = pair_target.shift(1, fill_value="CASH")
    pair_previous = pair_position.shift(1, fill_value="CASH")
    pair_overnight = pd.Series(0.0, index=prices.index)
    pair_intraday = pd.Series(0.0, index=prices.index)
    for asset in [risk, bond]:
        asset_close = prices[f"{asset}_adj_close"]
        asset_open = prices[f"{asset}_open"] * asset_close / prices[f"{asset}_close"]
        held_overnight = pair_previous.eq(asset)
        held_intraday = pair_position.eq(asset)
        pair_overnight.loc[held_overnight] = (asset_open / asset_close.shift(1) - 1).loc[held_overnight]
        pair_intraday.loc[held_intraday] = (asset_close / asset_open - 1).loc[held_intraday]
    pair_return = (1 + pair_overnight) * (1 + pair_intraday) - 1
    assert pair_return.notna().all()
    russell_results[name] = pd.DataFrame({
        "spread": russell_spread,
        "rsi_3": russell_rsi,
        "close_signal_target": pair_target,
        "position_after_open": pair_position,
        "strategy_return_before_costs": pair_return,
        "equity": (1 + pair_return).cumprod(),
    })
    russell_trades[name] = pd.DataFrame({
        "signal_date": pd.Series(prices.index, index=prices.index).shift(1),
        "execution_date": prices.index,
        "from": pair_previous,
        "to": pair_position,
    }).loc[pair_position.ne(pair_previous)]
    assert (russell_trades[name].signal_date < russell_trades[name].execution_date).all()


In [ ]:
for name, (risk, bond, *thresholds) in russell_tests.items():
    pair_comparison = pd.DataFrame({
        f"{name} strategy": russell_results[name]["equity"],
        f"{risk} buy and hold": prices[f"{risk}_adj_close"] / prices[f"{risk}_adj_close"].iloc[0],
        f"{bond} buy and hold": prices[f"{bond}_adj_close"] / prices[f"{bond}_adj_close"].iloc[0],
    })
    plot_equity_drawdown(pair_comparison, f"{name}: IWM/TLT signal, before costs, cash earns 0%")


## Performance metrics: all six strategies, before costs; 252 sessions/year; zero risk-free rate

In [ ]:
strategy_returns = pd.DataFrame({
    "QQQ / TLT": strategy_return,
    "QLD / UBT": leveraged_strategy_return,
    "USD / UBT": sector_results["USD"]["strategy_return_before_costs"],
    "ROM / UBT": sector_results["ROM"]["strategy_return_before_costs"],
    "IWM / TLT": russell_results["IWM / TLT"]["strategy_return_before_costs"],
    "UWM / UBT": russell_results["UWM / UBT"]["strategy_return_before_costs"],
})
strategy_positions = pd.DataFrame({
    "QQQ / TLT": position,
    "QLD / UBT": leveraged_position,
    "USD / UBT": sector_results["USD"]["position_after_open"],
    "ROM / UBT": sector_results["ROM"]["position_after_open"],
    "IWM / TLT": russell_results["IWM / TLT"]["position_after_open"],
    "UWM / UBT": russell_results["UWM / UBT"]["position_after_open"],
})
strategy_equities = (1 + strategy_returns).cumprod()
years = (prices.index[-1] - prices.index[0]).days / 365.25
metric_rows = {}
for name, returns in strategy_returns.items():
    growth = strategy_equities[name]
    dd = drawdown(growth)
    cagr = growth.iloc[-1] ** (1 / years) - 1
    volatility = returns.std(ddof=1) * np.sqrt(252)
    downside = np.sqrt(returns.clip(upper=0).pow(2).mean()) * np.sqrt(252)
    held = strategy_positions[name]
    previous = held.shift(1, fill_value="CASH")
    metric_rows[name] = {
        "Total return": growth.iloc[-1] - 1,
        "CAGR": cagr,
        "Annualized volatility": volatility,
        "Sharpe": returns.mean() * 252 / volatility if volatility > 0 else np.nan,
        "Sortino": returns.mean() * 252 / downside if downside > 0 else np.nan,
        "Max drawdown": dd.min(),
        "Calmar": cagr / abs(dd.min()) if dd.min() < 0 else np.nan,
        "Positive days (all sessions)": returns.gt(0).mean(),
        "Exposure (any part of session)": (held.ne("CASH") | previous.ne("CASH")).mean(),
        "Entries": int((held.ne(previous) & held.ne("CASH")).sum()),
    }
performance_metrics = pd.DataFrame.from_dict(metric_rows, orient="index")
percent_metrics = ["Total return", "CAGR", "Annualized volatility", "Max drawdown",
                   "Positive days (all sessions)", "Exposure (any part of session)"]
formats = {column: "{:.2%}" for column in percent_metrics}
formats.update({column: "{:.2f}" for column in ["Sharpe", "Sortino", "Calmar"]})
formats["Entries"] = "{:.0f}"
display(performance_metrics.style.format(formats))


## Monthly compounded strategy returns and yearly totals: shared color scale; boundary years may be partial

In [ ]:
monthly_returns = (1 + strategy_returns).resample("ME").prod(min_count=1) - 1
monthly_returns.index.name = "month"
years_shown = range(monthly_returns.index.year.min(), monthly_returns.index.year.max() + 1)
yearly_returns = (1 + monthly_returns).groupby(monthly_returns.index.year).prod(min_count=1) - 1
color_limit = max(float(monthly_returns.abs().max().max()), float(yearly_returns.abs().max().max()), 0.001)
fig, axes = plt.subplots(3, 2, figsize=(18, 21), constrained_layout=True)
for name, ax in zip(monthly_returns.columns, axes.flat):
    month_table = pd.DataFrame({
        "year": monthly_returns.index.year,
        "month": monthly_returns.index.month,
        "return": monthly_returns[name].to_numpy(),
    }).pivot(index="year", columns="month", values="return")
    month_table = month_table.reindex(index=years_shown, columns=range(1, 13))
    month_table["Year"] = yearly_returns[name].reindex(month_table.index)
    values = month_table.to_numpy()
    image = ax.imshow(np.ma.masked_invalid(values), cmap="RdYlGn", aspect="auto",
                      vmin=-color_limit, vmax=color_limit)
    ax.set_title(name)
    ax.set_xticks(range(13), ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                            "Jul", "Aug", "Sep", "Oct", "Nov", "Dec", "Year"])
    ax.axvline(11.5, color="black", linewidth=1)
    ax.set_yticks(range(len(month_table)), month_table.index)
    for row in range(values.shape[0]):
        for col in range(values.shape[1]):
            value = values[row, col]
            if np.isfinite(value):
                ax.text(col, row, f"{value:.1%}", ha="center", va="center", fontsize=8,
                        color="white" if abs(value) > 0.65 * color_limit else "black")
colorbar = fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.75)
colorbar.ax.yaxis.set_major_formatter(PercentFormatter(1))
colorbar.set_label("Compounded return, before costs")
plt.show()


## Test 05: RSI window and threshold grid; maximize full-sample Sharpe for each pair (in-sample, before costs)

In [ ]:
from itertools import product

RSI_WINDOWS = [2, 3, 4, 5, 7, 10, 14]
ENTRY_LOW = [5, 10, 15, 20, 25, 30]
ENTRY_HIGH = [70, 75, 80, 85, 90, 95]
EXIT_RISK = [50, 60, 70, 80]
EXIT_BOND = [20, 30, 40, 50]

grid_parameters = pd.DataFrame([
    (window, low, high, exit_risk, exit_bond)
    for window, low, high, exit_risk, exit_bond in product(
        RSI_WINDOWS, ENTRY_LOW, ENTRY_HIGH, EXIT_RISK, EXIT_BOND
    )
    if low < exit_bond <= exit_risk < high
], columns=["rsi_window", "entry_low", "entry_high", "exit_risk", "exit_bond"])
grid_tests = {
    "QQQ / TLT": (("QQQ", "TLT"), ("QQQ", "TLT")),
    "QLD / UBT": (("QLD", "UBT"), ("QQQ", "TLT")),
    "USD / UBT": (("USD", "UBT"), ("USD", "UBT")),
    "ROM / UBT": (("ROM", "UBT"), ("ROM", "UBT")),
    "IWM / TLT": (("IWM", "TLT"), ("IWM", "TLT")),
    "UWM / UBT": (("UWM", "UBT"), ("IWM", "TLT")),
}
print(f"{len(grid_parameters):,} configurations per test; {len(grid_parameters) * len(grid_tests):,} total")
print(f"Sample: {prices.index[0].date()} to {prices.index[-1].date()}")


In [ ]:
def evaluate_rsi_grid(data, assets, signal_assets, parameters):
    count = len(parameters)
    windows = sorted(parameters.rsi_window.unique())
    ratio = data[f"{signal_assets[0]}_adj_close"] / data[f"{signal_assets[1]}_adj_close"]
    rsi_values = np.array([wilder_rsi(ratio, int(window)).to_numpy() for window in windows])
    window_ids = parameters.rsi_window.map({window: i for i, window in enumerate(windows)}).to_numpy()
    entry_low = parameters.entry_low.to_numpy()
    entry_high = parameters.entry_high.to_numpy()
    exit_risk = parameters.exit_risk.to_numpy()
    exit_bond = parameters.exit_bond.to_numpy()
    asset_close = np.column_stack([data[f"{asset}_adj_close"] for asset in assets])
    asset_open = np.column_stack([
        data[f"{asset}_open"] * data[f"{asset}_adj_close"] / data[f"{asset}_close"]
        for asset in assets
    ])
    overnight = np.zeros((len(data), 3))
    intraday = np.zeros((len(data), 3))
    overnight[1:, 1:] = asset_open[1:] / asset_close[:-1] - 1
    intraday[:, 1:] = asset_close / asset_open - 1
    holding = np.zeros(count, dtype=int)
    equity_values = np.ones(count)
    peak = np.ones(count)
    max_dd = np.zeros(count)
    return_sum = np.zeros(count)
    return_square_sum = np.zeros(count)
    entries = np.zeros(count, dtype=int)
    exposed_sessions = np.zeros(count, dtype=int)
    for i in range(len(data)):
        previous = holding.copy()
        if i > 0:
            value = rsi_values[window_ids, i - 1]
            exit_now = ((holding == 1) & (value > exit_risk)) | ((holding == 2) & (value < exit_bond))
            holding[exit_now] = 0
            flat = holding == 0
            holding[flat & (value < entry_low)] = 1
            holding[flat & (value > entry_high)] = 2
        daily_return = (1 + overnight[i, previous]) * (1 + intraday[i, holding]) - 1
        return_sum += daily_return
        return_square_sum += daily_return ** 2
        equity_values *= 1 + daily_return
        peak = np.maximum(peak, equity_values)
        max_dd = np.minimum(max_dd, equity_values / peak - 1)
        entries += (holding != previous) & (holding != 0)
        exposed_sessions += (holding != 0) | (previous != 0)
    mean_return = return_sum / len(data)
    daily_std = np.sqrt(np.maximum(
        (return_square_sum - len(data) * mean_return ** 2) / (len(data) - 1), 0
    ))
    sharpe = np.full(count, np.nan)
    np.divide(mean_return * np.sqrt(252), daily_std, out=sharpe, where=daily_std > 0)
    elapsed_years = (data.index[-1] - data.index[0]).days / 365.25
    output = parameters.copy()
    output["Sharpe"] = sharpe
    output["CAGR"] = equity_values ** (1 / elapsed_years) - 1
    output["Annualized volatility"] = daily_std * np.sqrt(252)
    output["Max drawdown"] = max_dd
    output["Total return"] = equity_values - 1
    output["Exposure"] = exposed_sessions / len(data)
    output["Entries"] = entries
    return output

grid_results = pd.concat([
    evaluate_rsi_grid(prices, assets, signal_assets, grid_parameters).assign(test=name)
    for name, (assets, signal_assets) in grid_tests.items()
], ignore_index=True)
grid_results = grid_results.sort_values(
    ["test", "Sharpe", "rsi_window", "entry_low", "entry_high", "exit_risk", "exit_bond"],
    ascending=[True, False, True, True, True, True, True], na_position="last"
).reset_index(drop=True)
best_configs = grid_results.dropna(subset=["Sharpe"]).groupby("test", sort=False).head(1).set_index("test")
best_formats = {column: "{:.2%}" for column in [
    "CAGR", "Annualized volatility", "Max drawdown", "Total return", "Exposure"
]}
best_formats["Sharpe"] = "{:.3f}"
for column in ["rsi_window", "entry_low", "entry_high", "exit_risk", "exit_bond", "Entries"]:
    best_formats[column] = "{:.0f}"
display(best_configs.style.format(best_formats))
display(grid_results.groupby("test", sort=False).head(10).set_index("test").style.format(best_formats))
Path("data").mkdir(exist_ok=True)
grid_results.to_csv(Path("data") / "rsi_grid_results.csv", index=False)
best_configs.to_csv(Path("data") / "rsi_grid_best_configs.csv")


## In-sample best equity curves: selected by full-sample Sharpe; before costs, not out-of-sample results

In [ ]:
in_sample_best_results = {}
for name, config in best_configs.iterrows():
    (risk, bond), signal_assets = grid_tests[name]
    ratio = prices[f"{signal_assets[0]}_adj_close"] / prices[f"{signal_assets[1]}_adj_close"]
    values = wilder_rsi(ratio, int(config.rsi_window))
    chosen_target = pd.Series("CASH", index=prices.index)
    holding = "CASH"
    for date, value in values.items():
        if pd.notna(value):
            if holding == risk and value > config.exit_risk:
                holding = "CASH"
            elif holding == bond and value < config.exit_bond:
                holding = "CASH"
            if holding == "CASH":
                if value < config.entry_low:
                    holding = risk
                elif value > config.entry_high:
                    holding = bond
        chosen_target.loc[date] = holding
    chosen_position = chosen_target.shift(1, fill_value="CASH")
    previous = chosen_position.shift(1, fill_value="CASH")
    overnight_return = pd.Series(0.0, index=prices.index)
    intraday_return = pd.Series(0.0, index=prices.index)
    for asset in [risk, bond]:
        asset_close = prices[f"{asset}_adj_close"]
        asset_open = prices[f"{asset}_open"] * asset_close / prices[f"{asset}_close"]
        overnight_return.loc[previous.eq(asset)] = (asset_open / asset_close.shift(1) - 1).loc[previous.eq(asset)]
        intraday_return.loc[chosen_position.eq(asset)] = (asset_close / asset_open - 1).loc[chosen_position.eq(asset)]
    chosen_return = (1 + overnight_return) * (1 + intraday_return) - 1
    growth = (1 + chosen_return).cumprod()
    np.testing.assert_allclose(chosen_return.mean() / chosen_return.std() * np.sqrt(252), config.Sharpe, rtol=1e-10)
    np.testing.assert_allclose(growth.iloc[-1] - 1, config["Total return"], rtol=1e-10)
    np.testing.assert_allclose(drawdown(growth).min(), config["Max drawdown"], atol=1e-12)
    in_sample_best_results[name] = pd.DataFrame({
        "rsi": values,
        "close_signal_target": chosen_target,
        "position_after_open": chosen_position,
        "return_before_costs": chosen_return,
        "equity": growth,
        "drawdown": drawdown(growth),
    })
in_sample_best_equities = pd.DataFrame({name: frame.equity for name, frame in in_sample_best_results.items()})
in_sample_best_equities.to_csv(Path("data") / "rsi_grid_in_sample_best_equities.csv", index_label="date")


In [ ]:
fig = plt.figure(figsize=(18, 17), constrained_layout=True)
outer = fig.add_gridspec(3, 2)
for index, (name, config) in enumerate(best_configs.iterrows()):
    inner = outer[index // 2, index % 2].subgridspec(2, 1, height_ratios=[3, 1])
    equity_ax = fig.add_subplot(inner[0])
    dd_ax = fig.add_subplot(inner[1], sharex=equity_ax)
    risk = grid_tests[name][0][0]
    curves = pd.DataFrame({
        "In-sample best": in_sample_best_equities[name],
        "Original rules": strategy_equities[name],
        f"{risk} buy and hold": prices[f"{risk}_adj_close"] / prices[f"{risk}_adj_close"].iloc[0],
    })
    curves.plot(ax=equity_ax, logy=True, linewidth=1.2, grid=True)
    equity_ax.set_title(
        f"{name} | RSI {int(config.rsi_window)} | entries {int(config.entry_low)}/{int(config.entry_high)}"
        f" | exits {int(config.exit_risk)}/{int(config.exit_bond)}\n"
        f"In-sample best Sharpe: {config.Sharpe:.3f}"
    )
    equity_ax.set_ylabel("Growth of 1 (log)")
    equity_ax.tick_params(labelbottom=False)
    for column, line in zip(curves.columns, equity_ax.lines):
        dd_ax.plot(curves.index, drawdown(curves[column]), color=line.get_color(), linewidth=1)
    dd_ax.yaxis.set_major_formatter(PercentFormatter(1))
    dd_ax.set_ylabel("Drawdown")
    dd_ax.grid(alpha=0.3)
fig.suptitle("IN-SAMPLE BEST — selected by full-sample Sharpe\nNext-open execution; before costs; cash earns 0%", fontsize=16)
fig.savefig(Path("data") / "rsi_grid_in_sample_best_curves.png", dpi=150)
plt.show()


## Return attribution protocol: annual equity and bond contributions

Equity and bond contributions sum exactly to the strategy return. Overnight P&L belongs to the old holding; intraday P&L belongs to the new holding. Period contributions use wealth at each session start relative to wealth at period start. Cash earns 0%; returns are before costs. Boundary years may be partial.

In [ ]:
def attribute_pair(prices, positions, risk, bond, expected_returns):
    if not positions.index.equals(expected_returns.index) or not positions.index.equals(prices.index):
        raise ValueError("Prices, positions and returns must share the same dates")
    if not positions.isin(["CASH", risk, bond]).all():
        raise ValueError("Unknown holding in pair attribution")
    previous = positions.shift(1, fill_value="CASH")
    overnight = pd.DataFrame(0.0, index=prices.index, columns=["Equity", "Bonds"])
    intraday = overnight.copy()
    for asset, sleeve in [(risk, "Equity"), (bond, "Bonds")]:
        close = prices[f"{asset}_adj_close"]
        open_price = prices[f"{asset}_open"] * close / prices[f"{asset}_close"]
        overnight.loc[previous.eq(asset), sleeve] = (open_price / close.shift(1) - 1).loc[previous.eq(asset)]
        intraday.loc[positions.eq(asset), sleeve] = (close / open_price - 1).loc[positions.eq(asset)]
    daily = overnight + intraday.mul(1 + overnight.sum(axis=1), axis=0)
    np.testing.assert_allclose(daily.sum(axis=1), expected_returns, rtol=1e-10, atol=1e-12)
    growth_before_session = (1 + expected_returns).cumprod().shift(1, fill_value=1.0)
    pnl = daily.mul(growth_before_session, axis=0)
    output = {"daily": daily.assign(Total=expected_returns)}
    for label, frequency in [("monthly", "ME"), ("yearly", "YE")]:
        contributions = pnl.resample(frequency).sum(min_count=1).div(
            growth_before_session.resample(frequency).first(), axis=0
        )
        total = (1 + expected_returns).resample(frequency).prod(min_count=1) - 1
        np.testing.assert_allclose(contributions.sum(axis=1), total, rtol=1e-10, atol=1e-12)
        output[label] = contributions.assign(Total=total)
    output["full_sample"] = pnl.sum().to_frame().T
    output["full_sample"]["Total"] = (1 + expected_returns).prod() - 1
    np.testing.assert_allclose(
        output["full_sample"][["Equity", "Bonds"]].sum(axis=1),
        output["full_sample"].Total, rtol=1e-10, atol=1e-12
    )
    return output

def plot_pair_attribution(attribution, risk, bond, title, ax):
    table = attribution["yearly"]
    x = np.arange(len(table))
    equity_values = table.Equity.to_numpy()
    bond_values = table.Bonds.to_numpy()
    bottom = np.where(equity_values * bond_values > 0, equity_values, 0.0)
    ax.bar(x, equity_values, color="tab:blue", label=f"Equity ({risk})")
    ax.bar(x, bond_values, bottom=bottom, color="tab:orange", label=f"Bonds ({bond})")
    ax.scatter(x, table.Total, color="black", marker="D", s=16, label="Strategy return", zorder=3)
    ax.axhline(0, color="gray", linewidth=0.7)
    ax.set_xticks(x, table.index.year, rotation=60)
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.set_ylabel("Contribution (percentage points)")
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.2)
    ax.legend(fontsize=8)


In [ ]:
attribution_dir = Path("data") / "return_attribution"
attribution_dir.mkdir(parents=True, exist_ok=True)
baseline_attributions = {
    name: attribute_pair(prices, strategy_positions[name], *grid_tests[name][0], strategy_returns[name])
    for name in strategy_returns.columns
}
fig, axes = plt.subplots(3, 2, figsize=(18, 14), constrained_layout=True)
for (name, attribution), ax in zip(baseline_attributions.items(), axes.flat):
    risk, bond = grid_tests[name][0]
    plot_pair_attribution(attribution, risk, bond, f"{name}: original rules", ax)
fig.suptitle("Annual strategy return contributions: equity and bonds; before costs", fontsize=15)
fig.savefig(attribution_dir / "test_01_baseline_contributions.png", dpi=140)
plt.show()
for period in ["daily", "monthly", "yearly", "full_sample"]:
    table = pd.concat({name: result[period] for name, result in baseline_attributions.items()},
                      names=["strategy", "date"])
    table.to_csv(attribution_dir / f"test_01_baseline_{period}.csv")
display(pd.concat({name: result["full_sample"] for name, result in baseline_attributions.items()},
                  names=["strategy", "row"]).droplevel("row").style.format("{:.2%}"))


## In-sample best return attribution: selected by full-sample Sharpe

In [ ]:
best_attributions = {
    name: attribute_pair(prices, result.position_after_open, *grid_tests[name][0], result.return_before_costs)
    for name, result in in_sample_best_results.items()
}
fig, axes = plt.subplots(3, 2, figsize=(18, 14), constrained_layout=True)
for (name, attribution), ax in zip(best_attributions.items(), axes.flat):
    risk, bond = grid_tests[name][0]
    plot_pair_attribution(attribution, risk, bond, f"{name}: IN-SAMPLE BEST", ax)
fig.suptitle("IN-SAMPLE BEST: annual equity and bond return contributions; before costs", fontsize=15)
fig.savefig(attribution_dir / "test_01_in_sample_best_contributions.png", dpi=140)
plt.show()
for period in ["daily", "monthly", "yearly", "full_sample"]:
    table = pd.concat({name: result[period] for name, result in best_attributions.items()},
                      names=["strategy", "date"])
    table.to_csv(attribution_dir / f"test_01_in_sample_best_{period}.csv")
display(pd.concat({name: result["full_sample"] for name, result in best_attributions.items()},
                  names=["strategy", "row"]).droplevel("row").style.format("{:.2%}"))
